In [13]:
import ROOT as r
import fedrarootlogon 
from matplotlib import pyplot as plt
import matplotlib
import awkward as ak
import uproot
import numpy as np
import sys
sys.path.insert(0, "/home/baronunix/Scripts/")
from Clustering_Cosmici_Frammenti import *
import time


def g_func(x, p1, p2, p3):
    return p1* np.exp(- (x - p2)**2 / (2*p3**2) )
brick_id = "GSI2"
r.gStyle.SetOptStat("em")
PLOTS=0
%jsroot on

## Calculate VP123

In [8]:
principal = r.TPrincipal(3, "ND")

file_pca = r.TFile("PCA_123.root", "READ")
info_pca = file_pca.Get("pca")

for track in info_pca:
    vr0, vr1, vr2 = track.VR1_av, track.VR2_av, track.VR3_av
    vrs = np.zeros(3)
    vrs[0] = vr0
    vrs[1] = vr1
    vrs[2] = vr2
    principal.AddRow(vrs)
    
principal.MakePrincipals()
principal.MakeCode()

r.gInterpreter.ProcessLine('.L pca.C+')

0

Writing on file "pca.C" ... done


Info in <ACLiC>: modified script has already been compiled and loaded
Info in <ACLiC>: it will be regenerated and reloaded!
Info in <TUnixSystem::ACLiC>: creating shared library /home/baronunix/Scripts/GSI2/pca_C.so
Warning in cling::IncrementalParser::CheckABICompatibility():
  Possible C++ standard library mismatch, compiled with __GLIBCXX__ '20220324'
  Extraction of runtime standard library version was: '20220421'


In [9]:
r.gSystem.Load("pca_C.so")
vr123s, vr123b = [], []
for track in info_pca:
    vr2, vr3, vr0 = track.VR1_av, track.VR2_av, track.VR3_av
    vrs = np.zeros(3)
    vrs[0] = vr2
    vrs[1] = vr3
    vrs[2] = vr0
    princ = np.zeros(3)
    principal.X2P(vrs, princ)
    vr123s.append(princ)

vr123, vr123b = [], []
for i in vr123s:
    vr123.append(i[0])
    vr123b.append(i[1])
    
file_pca.Close()

In [10]:
file_pca_2 = r.TFile("PCA2.root", "RECREATE")

pca_1 = r.TNtuple("pca_1", "", "VR123:VR123b")

for i in range(len(vr123)):
    pca_1.Fill(vr123[i], vr123b[i])

pca_1.Write("pca")

429

In [65]:
pca_1.Draw("VR123>>v123s(125, -4., 7.)", "", "COLZ")

r.gStyle.SetOptStat(0)
histos = r.gDirectory.Get("v123s")
histos.SetTitle("VP_{123} [GSI2]; VP_{123}; Entries")
fit_func = r.TF1("fit_func", "[0]*TMath::Gaus(x, [1], [2]) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8])", -4, 7)
fit_func.SetParameters(300, -1.5, .5, 150, 1., 0.5, 30, 3., 0.5)

#ampiezze
fit_func.SetParLimits(0, 80, 300)
fit_func.SetParLimits(3, 20, 200)
fit_func.SetParLimits(6, 10, 70)

#punto medio
fit_func.SetParLimits(1, -1.4, -1.)
fit_func.SetParLimits(4, -0.5, 2.)
fit_func.SetParLimits(7, 2.5, 3.5)

#deviazione_st
fit_func.SetParLimits(2, 0., 1.)
fit_func.SetParLimits(5, 0., 1.)
fit_func.SetParLimits(8, 0., 1.)

tr = -1.8
histos.Fit("fit_func", "S", "", tr, 7)
histos.GetFunction("fit_func").SetLineColor(2)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Gaus(x, [1], [2])", tr, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(95)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

comp4 = r.TF1("comp4", "[0]*TMath::Gaus(x, [1], [2])", -4., tr)
comp4.SetParameters(params[0], params[1], params[2])
comp4.SetLineColor(4)
comp4.SetLineStyle(2)
comp4.Draw("SAME")

legend = r.TLegend(0.6,0.55,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Overall Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()), "")
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()), "")
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 4)), "")
legend.Draw("SAME")

line = r.TLine(tr, 0, tr, 563)
line.SetLineColor(12)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

t1 = r.TText(10000, 5000, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")

c.Draw()

 FCN=87.7077 FROM MIGRAD    STATUS=CONVERGED     846 CALLS         847 TOTAL
                     EDM=1.70234e-06    STRATEGY= 1      ERROR MATRIX ACCURATE 
  EXT PARAMETER                                   STEP         FIRST   
  NO.   NAME      VALUE            ERROR          SIZE      DERIVATIVE 
   1  p0           1.64351e+02   4.58134e+00   1.58572e-04   1.39868e-02
   2  p1          -1.36507e+00   4.27297e-02   9.92414e-04   5.06879e-03
   3  p2           7.16499e-01   6.53504e-02   2.01082e-04   7.75490e-03
   4  p3           8.20249e+01   4.68960e+00   1.06365e-04  -6.50814e-03
   5  p4           6.25907e-01   1.08309e-01   1.20016e-04  -3.20030e-02
   6  p5           1.00000e+00   6.17461e-01   4.55855e-03** at limit **
   7  p6           3.62610e+01   2.29970e+00   2.45866e-04  -8.21350e-03
   8  p7           3.03097e+00   6.24172e-02   2.88446e-04  -4.13048e-03
   9  p8           6.41473e-01   5.14220e-02   2.28285e-04  -1.42751e-02


Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 


In [69]:
pca_1.Draw("VR123>>v123s(125, -4., 7.)", "", "COLZ")

r.gStyle.SetOptStat(0)
histos = r.gDirectory.Get("v123s")
histos.SetTitle("VP_{123} [GSI2]; VP_{123}; Entries")
fit_func = r.TF1("fit_func", "[0]*TMath::Landau(x,[1],[2],0) + [3]*TMath::Gaus(x, [4], [5]) + [6]*TMath::Gaus(x, [7], [8])", -4, 7)
fit_func.SetParameters(300, -1.5, .5, 150, 1., 0.5, 30, 3., 0.5)

#ampiezze
fit_func.SetParLimits(0, 200, 3000)
fit_func.SetParLimits(3, 20, 200)
fit_func.SetParLimits(6, 10, 70)

#punto medio
fit_func.SetParLimits(1, -2., 2.)
fit_func.SetParLimits(4, -0.5, 2.)
fit_func.SetParLimits(7, 2.5, 3.5)

#deviazione_st
fit_func.SetParLimits(2, 0., 5.)
fit_func.SetParLimits(5, 0., 2.)
fit_func.SetParLimits(8, 0., 1.)

tr = -3.5
histos.Fit("fit_func", "S", "", tr, 6)
histos.GetFunction("fit_func").SetLineColor(2)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Landau(x,[1],[2],0)", tr, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
#comp1.Draw("SAME")

x = np.linspace(tr, 6., 1000)
gr = r.TGraph()
npoint = 0
for xpos in x:
    y_pos = comp1.Eval(xpos)
    gr.SetPoint(npoint,xpos,y_pos)
    npoint = npoint + 1
gr.SetMarkerColor(4)
gr.SetMarkerSize(1)
gr.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(95)
comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
comp3.Draw("SAME")

comp4 = r.TF1("comp4", "[0]*TMath::Gaus(x, [1], [2])", -4., tr)
comp4.SetParameters(params[0], params[1], params[2])
comp4.SetLineColor(4)
comp4.SetLineStyle(2)
comp4.Draw("SAME")

legend = r.TLegend(0.6,0.55,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Overall Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()), "")
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()), "")
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 4)), "")
legend.Draw("SAME")

line = r.TLine(tr, 0, tr, 563)
line.SetLineColor(12)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

t1 = r.TText(10000, 5000, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")



c.Draw()

 FCN=118.375 FROM MIGRAD    STATUS=CONVERGED     493 CALLS         494 TOTAL
                     EDM=6.57708e-08    STRATEGY= 1  ERROR MATRIX UNCERTAINTY   4.3 per cent
  EXT PARAMETER                                   STEP         FIRST   
  NO.   NAME      VALUE            ERROR          SIZE      DERIVATIVE 
   1  p0           8.59286e+02   2.73879e+01  -9.29918e-06   1.50281e-05
   2  p1          -1.44908e+00   2.14153e-02  -1.02238e-05  -3.53479e-03
   3  p2           3.72595e-01   1.30449e-02  -6.36824e-06  -1.44575e-02
   4  p3           5.81053e+01   3.48251e+00   3.45376e-05   4.53097e-03
   5  p4           4.22700e-01   7.11126e-02  -3.17245e-05   2.62361e-03
   6  p5           1.33217e+00   4.23528e-02  -1.95290e-05   5.55067e-04
   7  p6           3.05194e+01   2.81608e+00  -1.17968e-04  -2.46790e-04
   8  p7           3.10574e+00   4.02373e-02  -7.37304e-05  -6.44326e-03
   9  p8           4.09904e-01   4.32255e-02   1.27176e-04  -5.46653e-03


In [51]:
c = r.TCanvas()
comp1 = r.TF1("comp1", " TMath::Landau(x,[0],[1],0)", -5, 10)
comp1.SetParameter(0, 2.3)
comp1.SetParameter(1, 1.2)
comp1.Draw()
c.Draw()

In [67]:
comp1.Eval(-1.5)

161.46301085483344

In [57]:
pca_1.Draw("VR123>>v123s(80, -4., 7.)", "", "COLZ")

r.gStyle.SetOptStat(0)
histos = r.gDirectory.Get("v123s")
histos.SetTitle("VP_{123} [GSI2]; VP_{123}; Entries")
fit_func = r.TF1("fit_func", "[0]*TMath::Landau(x,[1],[2],0) + [3]*TMath::Landau(x, [4], [5], 0) + [6]*TMath::Gaus(x, [7], [8])", -4, 7)
fit_func.SetParameters(300, -1.5, .5, 150, 1., 0.5, 30, 3., 0.5)

#ampiezze
fit_func.SetParLimits(0, 200, 3000)
fit_func.SetParLimits(3, 20, 2000)
fit_func.SetParLimits(6, 10, 70)

#punto medio
fit_func.SetParLimits(1, -2., 2.)
fit_func.SetParLimits(4, -0.5, 2.)
fit_func.SetParLimits(7, 2.9, 3.5)

#deviazione_st
fit_func.SetParLimits(2, 0., 5.)
fit_func.SetParLimits(5, 0., 2.)
fit_func.SetParLimits(8, 0., 1.)

tr = -3.5
histos.Fit("fit_func", "S", "", tr, 6)
histos.GetFunction("fit_func").SetLineColor(2)

c = r.TCanvas()
histos.Draw("PE")

params = fit_func.GetParameters()

comp1 = r.TF1("comp1", "[0]*TMath::Landau(x,[1],[2],0)", tr, 6.)
comp1.SetParameters(params[0], params[1], params[2])
comp1.SetLineColor(4)
#comp1.Draw("SAME")

comp2 = r.TF1("comp2", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp2.SetParameters(params[3], params[4], params[5])
comp2.SetLineColor(95)
#comp2.Draw("SAME")

comp3 = r.TF1("comp3", "[0]*TMath::Gaus(x, [1], [2])", -3, 6.)
comp3.SetParameters(params[6], params[7], params[8])
comp3.SetLineColor(8)
#comp3.Draw("SAME")

comp4 = r.TF1("comp4", "[0]*TMath::Gaus(x, [1], [2])", -4., tr)
comp4.SetParameters(params[0], params[1], params[2])
comp4.SetLineColor(4)
comp4.SetLineStyle(2)
#comp4.Draw("SAME")

legend = r.TLegend(0.6,0.55,0.88,0.85)
legend.SetTextFont(0)
legend.SetTextSize(0.04)
#legend.AddEntry(histo, "VR123", "lpe")
legend.AddEntry(fit_func, "Overall Fit")
legend.AddEntry(comp1, "Z = 2")
legend.AddEntry(comp2, "Z = 3")
legend.AddEntry(comp3, "Z > 3")
legend.AddEntry("testo", "Entries = " + str(histos.GetEntries()), "")
legend.AddEntry("chi2 / NDF", "#chi^{2} / NDF = " + str(round(fit_func.GetChisquare(), 2)) + " / " + str(fit_func.GetNDF()), "")
legend.AddEntry("prob", "Prob = " + str(round(fit_func.GetProb(), 4)), "")
legend.Draw("SAME")

line = r.TLine(tr, 0, tr, 563)
line.SetLineColor(12)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

t1 = r.TText(10000, 5000, brick_id)
t1.SetTextColor(1)
t1.SetTextSize(20)
t1.Draw("SAME")



c.Draw()

 FCN=330.682 FROM MIGRAD    STATUS=CONVERGED     608 CALLS         609 TOTAL
                     EDM=3.72809e-08    STRATEGY= 1      ERROR MATRIX ACCURATE 
  EXT PARAMETER                                   STEP         FIRST   
  NO.   NAME      VALUE            ERROR          SIZE      DERIVATIVE 
   1  p0           1.60245e+03   3.47297e+01   1.57517e-04  -8.27981e-03
   2  p1          -1.33071e+00   1.82903e-02   7.69672e-05   8.36765e-03
   3  p2           4.34238e-01   1.06287e-02   3.43047e-05  -3.54456e-02
   4  p3           3.95235e+02   3.39543e+01   2.90948e-04  -2.85260e-03
   5  p4           7.44462e-01   7.96959e-02   4.48659e-04   3.22521e-04
   6  p5           2.37155e-01   2.53885e-02   2.11415e-04  -1.62483e-03
   7  p6           4.72285e+01   4.13963e+00   9.62864e-04  -5.09681e-04
   8  p7           2.90948e+00   3.38097e-01   4.86501e-03  -1.99818e-04
   9  p8           4.89508e-01   5.94009e-02   4.70349e-04  -6.06579e-04
